# 03 · Policy Inference and Evaluation / 闭环推理与评测
预计 40–50 分钟。Inference（推理）只计算动作，不更新模型；Rollout（闭环运行）在每次动作后获取新观测再计算下一动作。Success Rate（成功率）是完成任务的回合比例。必须执行自己的模型：标准 5 次、Position Shift Test 5 次。

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import display, Markdown, Image, Video
from manipulation_lab.common import ROOT, DATA, write_json
from manipulation_lab.notebook import workspace, command, next_stage, stage_path
GROUP = os.environ.get("LAB_GROUP", "group_01")  # 教师为每组分配；只改自己的组
RUN = workspace(GROUP)
TRAINING = stage_path(RUN, "training")
CHECKPOINT = TRAINING / "student_policy.pth"
print("Student checkpoint:", CHECKPOINT)

In [ ]:
from manipulation_lab.dataset import inspect
import yaml
info = inspect(DATA)
print("Training start-position minimum (world m):", info["train_initial_position_min_m"])
print("Training start-position maximum (world m):", info["train_initial_position_max_m"])
evaluation_config = yaml.safe_load((ROOT / "configs/evaluation.yaml").read_text())
print(evaluation_config)

两个条件使用同一组保留轨迹的初始机器人/物体状态、固定种子。位移只给 Can 加世界 X 方向 4 cm，并检查仍在源容器内；不声称是 OOD（训练分布之外）测试。标准集也没有用于训练。环境成功与额外稳定放置分别报告：目标区域内、张开夹爪、无夹爪接触，策略继续运行至少 1 秒且物体漂移不超过 1 cm、速度不超过 5 cm/s。抬起不等于成功。

In [ ]:
EVALUATION = next_stage(RUN, "evaluation")
command("evaluate.py", "--checkpoint", CHECKPOINT, "--config", ROOT / "configs/evaluation.yaml", "--output", EVALUATION, timeout=2700)
results = json.loads((EVALUATION / "summary.json").read_text())
assert all(c["episodes"] == 5 for c in results["conditions"].values())
print(json.dumps(results["conditions"], indent=2))
print("Evaluated checkpoint SHA256:", results["checkpoint_sha256"])
display(Image(filename=str(EVALUATION / "success_rate_plot.png")))

In [ ]:
for filename in ["successful_rollout.mp4", "failed_rollout.mp4"]:
    path = EVALUATION / filename
    if path.exists():
        display(Markdown("**" + filename + "**"))
        display(Video(str(path), embed=True))
    else:
        print(filename, "不存在：本次没有该类型回合，不能伪造视频。")

可选教师对照：下格默认关闭。打开后生成单独参考结果，绝不替换学生结果。5 次测试只能给出粗略估计，不应按绝对成功率排名。

In [ ]:
RUN_REFERENCE = False
if RUN_REFERENCE:
    reference = ROOT / "checkpoints/instructor_reference.pth"
    if not reference.exists():
        raise FileNotFoundError("教师尚未发布参考模型")
    reference_output = next_stage(RUN, "reference_evaluation")
    command("evaluate.py", "--checkpoint", reference, "--config", ROOT / "configs/evaluation.yaml", "--output", reference_output)
    print(json.loads((reference_output / "summary.json").read_text())["conditions"])